# Stage 1 — Synthetic SFT Warmup

**Goal:** Teach Llama-3.2-1B to output valid negotiation tool calls and basic negotiation grammar.  
**Runtime:** ~30 min on Colab T4.  
**Output:** SFT checkpoint pushed to HF Hub.

Run cells top-to-bottom. Colab T4 free tier is sufficient.

In [ ]:
# Install deps
!pip install -q unsloth trl datasets openai

In [ ]:
# Clone repo to get pre-generated SFT data (2K traces)
import os
if not os.path.exists('/content/AgentGrid_V1'):
    !git clone https://github.com/jayyyyqwq/GaN_J_AI.git /content/AgentGrid_V1
%cd /content/AgentGrid_V1
!pip install -q -e .

In [ ]:
import os, json, random
from pathlib import Path

# ── Config ──────────────────────────────────────────────────────────
BASE_MODEL     = "meta-llama/Llama-3.2-1B"
HF_REPO        = "Jayyyy234/agentgrid-sft"  # <- set your HF username
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")  # for trace generation
N_TRACES       = 2000
LORA_R         = 16
MAX_SEQ_LEN    = 2048
EPOCHS         = 2
OUTPUT_DIR     = "/content/sft_output"
DATA_PATH      = "/content/AgentGrid_V1/training/synthetic_traces/sft_data.jsonl"
# ────────────────────────────────────────────────────────────────────

## Step 1: Generate synthetic negotiation traces

Uses GPT-4o to generate 2000 realistic transcripts of 3-agent energy negotiation.

In [ ]:
from pathlib import Path
if Path(DATA_PATH).exists():
    print(f'Using pre-generated traces at {DATA_PATH}')
    traces = []
else:
    print('No pre-generated traces — regenerating via OpenAI')


In [ ]:
from pathlib import Path
if not Path(DATA_PATH).exists():
    raise RuntimeError('SFT data missing — re-run trace generation cell')
print(f'SFT data ready: {DATA_PATH}')


## Step 2: SFT with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=torch.float16,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_R,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing=True,
)
print("Model loaded.")

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

dataset = load_dataset("json", data_files=DATA_PATH, split="train")

def format_example(ex):
    return {"text": f"<s>[INST] {ex['prompt']} [/INST] {ex['completion']} </s>"}

dataset = dataset.map(format_example)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        save_strategy="epoch",
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
    ),
)

trainer.train()

## Step 3: Evaluate JSON validity rate

In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = (
    "<s>[INST] You are Agent A. Step 5. Battery: 0.62. Urgency: 0.81. "
    "Agent B has reputation 0.91. Choose ONE action tool. [/INST]"
)

valid_count = 0
N_EVAL = 50
for _ in range(N_EVAL):
    inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=80, do_sample=True, temperature=0.7)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    completion = text[len(test_prompt):].strip()
    try:
        json.loads(completion)
        valid_count += 1
    except json.JSONDecodeError:
        pass

validity_rate = valid_count / N_EVAL
print(f"JSON validity rate: {validity_rate:.1%}")
assert validity_rate >= 0.7, f"Validity rate {validity_rate:.1%} too low — check traces."

## Step 4: Push checkpoint to HF Hub

In [ ]:
from huggingface_hub import login
login()  # paste HF token when prompted

model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)
print(f"Checkpoint saved to hf.co/{HF_REPO}")